In [1]:
# Setup
import sys
sys.path.append('../src')

import requests
import pandas as pd
import time
from tqdm import tqdm

from utils.config import CARTOLA_BASE_URL, CARTOLA_SEASONS, BRONZE_CARTOLA
from utils.logger import setup_logger

logger = setup_logger('cartola', 'logs/cartola.log')
print("✅ Setup completo!")

✅ Setup completo!


In [2]:
# Teste rápido
url = f"{CARTOLA_BASE_URL}/atletas/mercado"
resp = requests.get(url)

if resp.status_code == 200:
    data = resp.json()
    print(f"✅ API funcionando! {len(data.get('atletas', []))} atletas")
else:
    print(f"❌ Erro {resp.status_code}")

✅ API funcionando! 689 atletas


In [3]:
# Extração principal
all_data = []

for season in CARTOLA_SEASONS:
    for rodada in tqdm(range(1, 39), desc=f"Temporada {season}"):
        url = f"{CARTOLA_BASE_URL}/atletas/pontuados/{rodada}"
        
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                
                for aid, atleta in data.get('atletas', {}).items():
                    atleta['temporada'] = season
                    atleta['rodada'] = rodada
                    all_data.append(atleta)
            
            time.sleep(1)  # Rate limit
        
        except:
            continue

df = pd.DataFrame(all_data)
print(f"\n✅ {len(df):,} registros extraídos")

Temporada 2024: 100%|██████████| 38/38 [00:42<00:00,  1.12s/it]


✅ 1,680 registros extraídos


In [4]:
# Salvar
BRONZE_CARTOLA.mkdir(parents=True, exist_ok=True)

output = BRONZE_CARTOLA / 'atletas_all.parquet'
df.to_parquet(output, compression='snappy', index=False)

print(f"💾 Salvo: {output}")
print(f"📊 Shape: {df.shape}")
print(f"📦 Tamanho: {output.stat().st_size / 1024 / 1024:.2f} MB")

💾 Salvo: D:\football analytics project\data\bronze\cartola\atletas_all.parquet
📊 Shape: (1680, 9)
📦 Tamanho: 0.02 MB
